# ATLAS orography and aspect processing

This notebook prepares terrain variables required by the ATLAS workflow for Chile.

It processes two elevation data sources:

1. **ERA5 Land geopotential**, converted to orography and aspect using the standard ERA5 Land method from the solar notebook.
2. **GLO 90 DEM**, processed separately for `continental` and `islands` to reduce memory usage.

Important distinction:

* **ERA5 Land is processed once for Chile**, using the dissolved Chile geometry from `REGIONES_v1.shp`. It is **not** split into `continental` and `islands`.
* **GLO 90 is processed separately for `continental` and `islands`**, because the higher resolution raster can fill the memory during aspect calculation.

Final ERA5 Land outputs:

```text
era5land_orography_chile.nc
era5land_aspect_chile.nc
```

Final GLO 90 outputs:

```text
glo90_orography_chile.nc
glo90_aspect_chile.nc
```


## Step 1. User parameters

Edit this cell before running the notebook.

Expected input files inside `./DEMdata/{country}/`:

```text
geo_1279l4_0.1x0.1.grib2_v4_unpack.nc
```

The Copernicus GLO-90 DEM is **not included** in the repository and must be downloaded separately from:

https://portal.opentopography.org/raster?opentopoID=OTSDEM.032021.4326.1

To download the DEM:

1. Open the OpenTopography portal.
2. Select the grid tile(s) corresponding to the country of interest.
3. Keep **GeoTIFF** as the output format.
4. Under **Raster Visualization**, select **Aspect**.
5. Download the resulting GeoTIFF file(s).

If the GLO-90 DEM is downloaded as multiple tiles, list their paths in `glo90_tile_paths`. The notebook will merge them automatically before processing.

For the country boundary, the recommended option for Chile is to use the regional shapefile `REGIONES_v1.shp`. The notebook will dissolve all regional geometries into a single country geometry and reproject it to EPSG:4326 before cropping.

The Natural Earth global shapefile remains available as a fallback for other countries.


## Important note on ERA5 Land longitudes

ERA5 Land files may use longitudes in the `0..360` convention. The Chile boundary loaded from `REGIONES_v1.shp` is reprojected to `EPSG:4326` and uses the standard `-180..180` convention.

Before writing the temporary ERA5 Land GeoTIFFs and before clipping to Chile, this notebook normalizes ERA5 Land longitudes to `-180..180`. This avoids `NoDataInBounds` errors during `rioxarray.clip()`.

If you already generated temporary ERA5 Land GeoTIFFs with the previous version of the notebook, set:

```python
overwrite = True
```

and re-run the ERA5 Land section so that the temporary files are regenerated with the corrected longitude convention.

In [1]:
from pathlib import Path

# Country name used in file names and folder names.
country = "chile"

# Main folder containing DEM and output files.
dem_path = Path(f"../DEMdata/{country}")
input_path = dem_path
output_path = dem_path
output_path.mkdir(parents=True, exist_ok=True)

# ERA5 Land geopotential input.
# This file is converted from geopotential to orography using: orography = geopotential / 9.81.
era5_geopotential_path = Path("../DEMdata/geo_1279l4_0.1x0.1.grib2_v4_unpack.nc")

# Input GLO 90 GeoTIFF files.
# Update these paths if your files have different names.
glo90_units = {
    "continental": input_path / f"glo90_orography_{country}_continental.tif",
    "islands": input_path / f"glo90_orography_{country}_islands.tif",
}

# Country boundary shapefile.
# For Chile, REGIONES_v1.shp contains regional polygons. They are dissolved into one country geometry.
use_country_boundary = True
local_boundary_path = Path("../data/shapefiles/chile/REGIONES_v1.shp")

# Output variable names.
orography_variable_name = "z"
aspect_variable_name = "aspect"

# Processing switches.
run_era5_land = True
run_glo90_orography_netcdf = True
run_glo90_aspect = True
run_glo90_final_merge = True

# If True, existing outputs are overwritten.
overwrite = True

# Aspect method.
# Recommended: "auto". The notebook first tries GDAL gdaldem.
# If gdaldem is not available, it uses a memory safe rasterio block by block fallback.
# Other accepted values: "gdal" or "rasterio_blockwise".
aspect_method = "auto"

# Tile size used only by the rasterio blockwise aspect fallback.
# Lower this value if memory is still high.
aspect_tile_size = 2048

# Compression level used when saving NetCDF files.
netcdf_compression_level = 4

# Dask chunks used when opening NetCDF files for the final GLO 90 merge.
# Smaller chunks reduce peak memory, but can make writing slower.
merge_chunks = {"latitude": 2048, "longitude": 2048}

# Print detailed progress and raster information.
verbose = True


## Step 2. Imports

Run this cell once. The notebook requires a geospatial Python environment with `geopandas`, `rasterio`, `rioxarray`, `xarray` and `netCDF4`.

For aspect calculation, the notebook first tries the GDAL command line tool `gdaldem`. If it is not available, it falls back to a memory safe `rasterio` block by block implementation.


In [2]:
import gc
import os
import shutil
import subprocess
import time
import warnings

import geopandas as gpd
import numpy as np
import rasterio
import rioxarray
import xarray as xr

warnings.filterwarnings("ignore", category=FutureWarning)

## Step 3. Helper functions

These functions keep the workflow readable and avoid repeating the same code for `continental` and `islands`.

In [3]:
def print_step(message):
    """Print a clear progress message."""
    print(f"[{time.strftime('%H:%M:%S')}] {message}", flush=True)


def print_memory(prefix="Memory"):
    """Print current Python process memory if psutil is available."""
    try:
        import psutil
        process = psutil.Process(os.getpid())
        memory_gb = process.memory_info().rss / 1024**3
        print(f"{prefix}: {memory_gb:.2f} GB used by the current Python process", flush=True)
    except Exception:
        print(f"{prefix}: psutil is not available, memory usage not printed", flush=True)


def print_raster_info(path, label="Raster"):
    """Print useful raster information without loading the raster into memory."""
    path = Path(path)
    if not path.exists():
        print(f"{label}: file not found: {path}", flush=True)
        return

    size_gb = path.stat().st_size / 1024**3
    with rasterio.open(path) as src:
        print(f"{label}: {path}", flush=True)
        print(f"  File size: {size_gb:.2f} GB", flush=True)
        print(f"  CRS: {src.crs}", flush=True)
        print(f"  Width x height: {src.width} x {src.height}", flush=True)
        print(f"  Resolution: {src.res}", flush=True)
        print(f"  Bounds: {src.bounds}", flush=True)
        print(f"  Data type: {src.dtypes[0]}", flush=True)
        approx_array_gb = src.width * src.height * np.dtype(src.dtypes[0]).itemsize / 1024**3
        print(f"  Approx. one band array size in memory: {approx_array_gb:.2f} GB", flush=True)


def should_write(path, overwrite=True):
    """Return True when an output file should be written."""
    path = Path(path)
    return overwrite or not path.exists()


def rename_coordinates(ds):
    """Rename common spatial coordinate names to longitude and latitude."""
    rename_map = {}

    if "x" in ds.coords:
        rename_map["x"] = "longitude"
    if "y" in ds.coords:
        rename_map["y"] = "latitude"
    if "lon" in ds.coords:
        rename_map["lon"] = "longitude"
    if "lat" in ds.coords:
        rename_map["lat"] = "latitude"

    if rename_map:
        ds = ds.rename(rename_map)

    return ds


def drop_auxiliary_coordinates(ds):
    """Remove auxiliary coordinates that are not needed in the final NetCDF files."""
    for coord in ["spatial_ref", "band"]:
        if coord in ds.coords:
            ds = ds.drop_vars(coord)
    return ds.squeeze(drop=True)



def normalize_longitudes_to_180(ds):
    """Convert longitude coordinates from 0..360 to -180..180 when needed.

    ERA5 and ERA5 Land files are sometimes stored with longitudes in the 0..360
    convention. The Chile boundary shapefile is in EPSG:4326 using -180..180.
    If the conventions are different, rioxarray.clip() can return NoDataInBounds.
    """
    if "longitude" not in ds.coords:
        return ds

    lon = ds["longitude"]
    try:
        lon_min = float(lon.min())
        lon_max = float(lon.max())
    except Exception:
        return ds

    print(f"Longitude range before normalization: {lon_min:.4f} to {lon_max:.4f}", flush=True)

    if lon_max > 180.0:
        print_step("Converting longitudes from 0..360 to -180..180")
        ds = ds.assign_coords(longitude=(((ds.longitude + 180) % 360) - 180))
        ds = ds.sortby("longitude")
        lon_min = float(ds.longitude.min())
        lon_max = float(ds.longitude.max())
        print(f"Longitude range after normalization: {lon_min:.4f} to {lon_max:.4f}", flush=True)

    return ds


def print_dataset_spatial_bounds(ds, label="Dataset"):
    """Print approximate spatial bounds of an xarray dataset."""
    if "longitude" in ds.coords and "latitude" in ds.coords:
        print(
            f"{label} bounds: "
            f"lon {float(ds.longitude.min()):.4f} to {float(ds.longitude.max()):.4f}, "
            f"lat {float(ds.latitude.min()):.4f} to {float(ds.latitude.max()):.4f}",
            flush=True,
        )


def geometries_overlap_raster(ds, geometry):
    """Check whether dataset coordinate bounds overlap the geometry bounds."""
    if geometry is None:
        return True
    if "longitude" not in ds.coords or "latitude" not in ds.coords:
        return True

    raster_bounds = (
        float(ds.longitude.min()),
        float(ds.latitude.min()),
        float(ds.longitude.max()),
        float(ds.latitude.max()),
    )
    geom_bounds = tuple(float(v) for v in geometry.total_bounds)

    print(f"Raster bounds used for clip: {raster_bounds}", flush=True)
    print(f"Geometry bounds used for clip: {geom_bounds}", flush=True)

    return not (
        raster_bounds[2] < geom_bounds[0]
        or raster_bounds[0] > geom_bounds[2]
        or raster_bounds[3] < geom_bounds[1]
        or raster_bounds[1] > geom_bounds[3]
    )

def prepare_spatial_dataset(ds):
    """Prepare a raster dataset for NetCDF export and optional clipping."""
    ds = rename_coordinates(ds)
    ds = drop_auxiliary_coordinates(ds)
    ds = normalize_longitudes_to_180(ds)

    if "latitude" in ds.coords:
        ds["latitude"] = ds.latitude.astype("float32")
    if "longitude" in ds.coords:
        ds["longitude"] = ds.longitude.astype("float32")

    ds = ds.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude", inplace=False)
    ds = ds.rio.write_crs("EPSG:4326", inplace=False)
    print_dataset_spatial_bounds(ds, "Prepared spatial dataset")
    return ds


def save_netcdf_compressed(ds, path, complevel=4, use_float32=True):
    """Save an xarray Dataset to NetCDF using compression."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    encoding = {}
    for var in ds.data_vars:
        var_encoding = {
            "zlib": True,
            "complevel": complevel,
            "shuffle": True,
        }
        if use_float32 and np.issubdtype(ds[var].dtype, np.floating):
            var_encoding["dtype"] = "float32"
        encoding[var] = var_encoding

    ds.to_netcdf(path, engine="netcdf4", encoding=encoding)


def load_dissolved_boundary(path):
    """Load a local shapefile, reproject it to EPSG:4326 and dissolve all polygons."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Boundary file not found: {path}")

    gdf = gpd.read_file(path)
    if gdf.empty:
        raise ValueError(f"Boundary file contains no features: {path}")
    if gdf.crs is None:
        raise ValueError(f"Boundary file has no CRS: {path}")

    gdf = gdf.to_crs("EPSG:4326")
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    dissolved = gdf.dissolve().explode(index_parts=False).reset_index(drop=True)
    return dissolved.geometry


def clip_dataset_to_geometry(ds, geometry):
    """Clip an xarray Dataset using an EPSG:4326 geometry."""
    ds = prepare_spatial_dataset(ds)
    return ds.rio.clip(geometry, crs="EPSG:4326", all_touched=True)

## Step 4. Aspect and NetCDF conversion functions

The aspect calculation uses the safest available method:

1. `gdaldem aspect`, if the GDAL command line tools are installed.
2. A `rasterio` block by block fallback, if `gdaldem` is not available.

The fallback avoids loading the full DEM into RAM. It reads small windows from the raster, computes aspect for each window, and writes the result progressively to disk.

In [4]:

def _compute_aspect_array_horn(z, xres, yres, nodata_value=-9999.0):
    """Compute aspect for one numpy array using a Horn 3 x 3 neighbourhood.

    This function works on one block only. It is used by the rasterio blockwise fallback.
    Output convention follows the usual GIS convention: 0 is North, 90 is East,
    180 is South and 270 is West. Flat cells are set to 0.
    """
    z = z.astype("float32", copy=False)
    z = np.where(np.isfinite(z), z, np.nan)

    # Pad with edge values so that the output has the same shape as the input block.
    zp = np.pad(z, pad_width=1, mode="edge")

    z1 = zp[:-2, :-2]
    z2 = zp[:-2, 1:-1]
    z3 = zp[:-2, 2:]
    z4 = zp[1:-1, :-2]
    z6 = zp[1:-1, 2:]
    z7 = zp[2:, :-2]
    z8 = zp[2:, 1:-1]
    z9 = zp[2:, 2:]

    dzdx = ((z3 + 2 * z6 + z9) - (z1 + 2 * z4 + z7)) / (8 * xres)
    dzdy = ((z7 + 2 * z8 + z9) - (z1 + 2 * z2 + z3)) / (8 * yres)

    aspect = 90.0 - np.degrees(np.arctan2(dzdy, -dzdx))
    aspect = np.where(aspect < 0, aspect + 360.0, aspect)
    aspect = np.where(aspect >= 360.0, aspect - 360.0, aspect)

    flat = (np.abs(dzdx) < 1e-12) & (np.abs(dzdy) < 1e-12)
    aspect = np.where(flat, 0.0, aspect)
    aspect = np.where(np.isfinite(aspect), aspect, nodata_value)
    return aspect.astype("float32")


def compute_aspect_with_rasterio_blockwise(input_tif, output_tif, overwrite=True, tile_size=2048):
    """Compute aspect block by block using rasterio, without loading the full DEM into memory."""
    input_tif = Path(input_tif)
    output_tif = Path(output_tif)
    output_tif.parent.mkdir(parents=True, exist_ok=True)

    if not input_tif.exists():
        raise FileNotFoundError(f"Input DEM not found: {input_tif}")

    if not should_write(output_tif, overwrite):
        print_step(f"Skipping existing aspect GeoTIFF: {output_tif}")
        return output_tif

    print_step(f"Computing aspect block by block with rasterio for: {input_tif.name}")
    print("This fallback is slower than gdaldem, but it avoids loading the full DEM into RAM.", flush=True)
    print_raster_info(input_tif, "Aspect input")
    print_memory("Before rasterio blockwise aspect")

    nodata_out = -9999.0

    with rasterio.open(input_tif) as src:
        xres = abs(src.transform.a)
        yres = abs(src.transform.e)

        profile = src.profile.copy()
        profile.update(
            driver="GTiff",
            dtype="float32",
            count=1,
            nodata=nodata_out,
            compress="lzw",
            tiled=True,
            blockxsize=256,
            blockysize=256,
            BIGTIFF="YES",
        )

        width = src.width
        height = src.height
        total_tiles = ((height + tile_size - 1) // tile_size) * ((width + tile_size - 1) // tile_size)
        tile_counter = 0

        with rasterio.open(output_tif, "w", **profile) as dst:
            for row_off in range(0, height, tile_size):
                win_h = min(tile_size, height - row_off)
                for col_off in range(0, width, tile_size):
                    win_w = min(tile_size, width - col_off)
                    tile_counter += 1

                    read_row_off = max(row_off - 1, 0)
                    read_col_off = max(col_off - 1, 0)
                    read_row_end = min(row_off + win_h + 1, height)
                    read_col_end = min(col_off + win_w + 1, width)

                    read_window = rasterio.windows.Window(
                        read_col_off,
                        read_row_off,
                        read_col_end - read_col_off,
                        read_row_end - read_row_off,
                    )
                    write_window = rasterio.windows.Window(col_off, row_off, win_w, win_h)

                    arr = src.read(1, window=read_window, masked=True).astype("float32")
                    arr = arr.filled(np.nan)

                    aspect_full = _compute_aspect_array_horn(arr, xres=xres, yres=yres, nodata_value=nodata_out)

                    inner_row_off = row_off - read_row_off
                    inner_col_off = col_off - read_col_off
                    aspect_tile = aspect_full[
                        inner_row_off:inner_row_off + win_h,
                        inner_col_off:inner_col_off + win_w,
                    ]

                    dst.write(aspect_tile, 1, window=write_window)

                    if verbose and (tile_counter == 1 or tile_counter % 20 == 0 or tile_counter == total_tiles):
                        print(
                            f"  Processed tile {tile_counter}/{total_tiles} "
                            f"at row {row_off}, col {col_off}",
                            flush=True,
                        )
                        print_memory("  Current memory")

                    del arr, aspect_full, aspect_tile

    gc.collect()
    print_memory("After rasterio blockwise aspect")
    print_raster_info(output_tif, "Aspect output")
    return output_tif


def compute_aspect_with_gdal(input_tif, output_tif, overwrite=True):
    """Compute aspect using GDAL gdaldem."""
    input_tif = Path(input_tif)
    output_tif = Path(output_tif)
    output_tif.parent.mkdir(parents=True, exist_ok=True)

    if not input_tif.exists():
        raise FileNotFoundError(f"Input DEM not found: {input_tif}")

    if not should_write(output_tif, overwrite):
        print_step(f"Skipping existing aspect GeoTIFF: {output_tif}")
        return output_tif

    gdaldem_path = shutil.which("gdaldem")
    if gdaldem_path is None:
        raise RuntimeError("GDAL gdaldem was not found in this environment.")

    print_step(f"Computing aspect with GDAL for: {input_tif.name}")
    print_raster_info(input_tif, "Aspect input")
    print_memory("Before gdaldem aspect")

    cmd = [
        gdaldem_path,
        "aspect",
        str(input_tif),
        str(output_tif),
        "-of",
        "GTiff",
        "-compute_edges",
        "-zero_for_flat",
        "-co",
        "COMPRESS=LZW",
        "-co",
        "TILED=YES",
        "-co",
        "BIGTIFF=YES",
    ]

    print("Command:", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)

    print_memory("After gdaldem aspect")
    print_raster_info(output_tif, "Aspect output")
    return output_tif


def compute_aspect(input_tif, output_tif, method="auto", overwrite=True, tile_size=2048):
    """Compute aspect using GDAL when available, otherwise use a blockwise rasterio fallback."""
    method = method.lower()

    if method not in ["auto", "gdal", "rasterio_blockwise"]:
        raise ValueError("aspect_method must be one of: 'auto', 'gdal', 'rasterio_blockwise'.")

    if method in ["auto", "gdal"]:
        gdaldem_path = shutil.which("gdaldem")
        if gdaldem_path is not None:
            return compute_aspect_with_gdal(input_tif, output_tif, overwrite=overwrite)
        if method == "gdal":
            raise RuntimeError(
                "GDAL gdaldem was not found. Either install GDAL command line tools or set "
                "aspect_method = 'rasterio_blockwise'."
            )
        print_step("GDAL gdaldem was not found. Falling back to rasterio blockwise aspect.")

    return compute_aspect_with_rasterio_blockwise(
        input_tif=input_tif,
        output_tif=output_tif,
        overwrite=overwrite,
        tile_size=tile_size,
    )


def raster_to_netcdf(
    input_tif,
    output_nc,
    variable_name,
    geometry=None,
    overwrite=True,
    complevel=4,
):
    """Convert one GeoTIFF file to one compressed NetCDF file."""
    input_tif = Path(input_tif)
    output_nc = Path(output_nc)
    output_nc.parent.mkdir(parents=True, exist_ok=True)

    if not input_tif.exists():
        raise FileNotFoundError(f"Input GeoTIFF not found: {input_tif}")

    if not should_write(output_nc, overwrite):
        print_step(f"Skipping existing NetCDF: {output_nc}")
        return output_nc

    print_step(f"Opening raster for NetCDF export: {input_tif.name}")
    print_raster_info(input_tif, "NetCDF input")
    print_memory("Before opening raster")

    ds = rioxarray.open_rasterio(input_tif).to_dataset(name=variable_name)
    ds = prepare_spatial_dataset(ds)

    if geometry is not None:
        print_step("Applying optional boundary crop")
        if not geometries_overlap_raster(ds, geometry):
            raise ValueError(
                "No overlap between raster and geometry before clipping. "
                "This usually means the raster is still in 0..360 longitudes while the Chile boundary is in -180..180, "
                "or the ERA5 Land file does not cover Chile. Re-run the ERA5 Land cells with overwrite = True."
            )
        ds = ds.rio.clip(geometry, crs="EPSG:4326", all_touched=True, from_disk=True)

    ds = ds.where(ds != -9999, np.nan)
    ds = drop_auxiliary_coordinates(ds)

    print(f"Dataset dimensions: {dict(ds.sizes)}", flush=True)
    print_memory("Before saving NetCDF")
    print_step(f"Saving NetCDF: {output_nc}")

    save_netcdf_compressed(ds, output_nc, complevel=complevel)

    del ds
    gc.collect()
    print_memory("After saving NetCDF and cleaning variables")

    return output_nc


## Step 5. ERA5 Land processing functions

This section follows the standard methodology used in the solar orography notebook, with one important project specific detail: the crop is performed using the **Chile country geometry** obtained by dissolving all polygons in `REGIONES_v1.shp`.

ERA5 Land is processed as a single Chile product, without any `continental` or `islands` split.

Workflow:

1. Open the ERA5 Land geopotential file.
2. Convert geopotential to orography using `z / 9.81`.
3. Save the full ERA5 Land orography as a temporary GeoTIFF.
4. Compute aspect from the full ERA5 Land orography GeoTIFF.
5. Crop both orography and aspect to the dissolved Chile geometry when exporting the final NetCDF files.
6. Save `era5land_orography_chile.nc` and `era5land_aspect_chile.nc`.


In [5]:
def _select_first_data_variable(ds):
    """Return the first data variable name in a Dataset."""
    if not ds.data_vars:
        raise ValueError("The dataset contains no data variables.")
    return list(ds.data_vars)[0]


def process_era5_land_orography_and_aspect(
    era5_geopotential_path,
    output_path,
    country,
    country_geometry,
    overwrite=True,
    complevel=4,
):
    """Create ERA5 Land orography and aspect NetCDF files for the selected country.

    This function follows the standard ERA5 Land methodology used in the solar notebook.

    Workflow:
    1. Convert the full ERA5 Land geopotential raster to orography in metres.
    2. Save the full orography raster as a temporary GeoTIFF.
    3. Compute aspect from the full orography GeoTIFF.
    4. Crop to the dissolved Chile country geometry when exporting the final NetCDF files.

    The ERA5 Land input is geopotential. Orography in metres is computed as:

        orography = geopotential / 9.81
    """
    era5_geopotential_path = Path(era5_geopotential_path)
    output_path = Path(output_path)
    output_path.mkdir(parents=True, exist_ok=True)

    if not era5_geopotential_path.exists():
        raise FileNotFoundError(f"ERA5 Land geopotential file not found: {era5_geopotential_path}")

    era5_orography_nc = output_path / f"era5land_orography_{country}.nc"
    era5_aspect_nc = output_path / f"era5land_aspect_{country}.nc"

    # Temporary full-domain GeoTIFFs.
    # These follow the same standard approach used in the solar notebook.
    full_orography_tif = output_path / "era5land_orography_global_wgs84.tif"
    full_aspect_tif = output_path / "era5land_aspect_global_wgs84.tif"

    print_step("Starting ERA5 Land orography and aspect processing")
    print(f"ERA5 Land input: {era5_geopotential_path}", flush=True)
    print("ERA5 Land method: one Chile crop using the dissolved country geometry, no continental/islands split.", flush=True)
    print_memory("Before opening ERA5 Land geopotential")

    if should_write(full_orography_tif, overwrite):
        print_step("Opening ERA5 Land geopotential")
        ds = xr.open_dataset(era5_geopotential_path)
        var_name = _select_first_data_variable(ds)
        print(f"Using ERA5 Land variable: {var_name}", flush=True)

        # Convert geopotential to orography in metres.
        oro = (ds[var_name] / 9.81).to_dataset(name=orography_variable_name)
        oro = prepare_spatial_dataset(oro)
        oro = oro.where(oro != -9999, np.nan)
        oro = drop_auxiliary_coordinates(oro)

        print(f"Full ERA5 Land orography dimensions before country crop: {dict(oro.sizes)}", flush=True)
        print_step(f"Saving full ERA5 Land orography GeoTIFF: {full_orography_tif.name}")
        oro.rio.to_raster(
            full_orography_tif,
            compress="LZW",
            tiled=True,
            BIGTIFF="YES",
        )
        print_raster_info(full_orography_tif, "Full ERA5 Land orography GeoTIFF")

        ds.close()
        del ds, oro
        gc.collect()
        print_memory("After saving full ERA5 Land orography GeoTIFF")
    else:
        print_step(f"Skipping existing full ERA5 Land orography GeoTIFF: {full_orography_tif.name}")

    if should_write(full_aspect_tif, overwrite):
        print_step("Computing full ERA5 Land aspect from the full orography GeoTIFF")
        compute_aspect(
            input_tif=full_orography_tif,
            output_tif=full_aspect_tif,
            method=aspect_method,
            overwrite=overwrite,
            tile_size=aspect_tile_size,
        )
    else:
        print_step(f"Skipping existing full ERA5 Land aspect GeoTIFF: {full_aspect_tif.name}")

    if should_write(era5_orography_nc, overwrite):
        print_step(f"Exporting cropped ERA5 Land orography NetCDF: {era5_orography_nc.name}")
        raster_to_netcdf(
            input_tif=full_orography_tif,
            output_nc=era5_orography_nc,
            variable_name=orography_variable_name,
            geometry=country_geometry,
            overwrite=overwrite,
            complevel=complevel,
        )
    else:
        print_step(f"Skipping existing ERA5 Land orography NetCDF: {era5_orography_nc.name}")

    if should_write(era5_aspect_nc, overwrite):
        print_step(f"Exporting cropped ERA5 Land aspect NetCDF: {era5_aspect_nc.name}")
        raster_to_netcdf(
            input_tif=full_aspect_tif,
            output_nc=era5_aspect_nc,
            variable_name=aspect_variable_name,
            geometry=country_geometry,
            overwrite=overwrite,
            complevel=complevel,
        )
    else:
        print_step(f"Skipping existing ERA5 Land aspect NetCDF: {era5_aspect_nc.name}")

    print_step("Completed ERA5 Land processing")
    return {
        "era5land_orography": era5_orography_nc,
        "era5land_aspect": era5_aspect_nc,
    }


## Step 6. Process GLO 90 units separately

This cell creates four intermediate GLO 90 NetCDF files:

```text
glo90_orography_chile_continental.nc
glo90_orography_chile_islands.nc
glo90_aspect_chile_continental.nc
glo90_aspect_chile_islands.nc
```

The aspect calculation is run separately for each input GeoTIFF.


In [6]:
if use_country_boundary:
    country_geometry = load_dissolved_boundary(local_boundary_path)
    print_step("Chile country geometry loaded for country crop")
    print("The geometry was created by dissolving all polygons from REGIONES_v1.shp.", flush=True)
    print(f"Chile geometry bounds in EPSG:4326: {country_geometry.total_bounds}", flush=True)
else:
    country_geometry = None
    print_step("Country boundary crop disabled")

outputs = {}

if run_era5_land:
    outputs.update(
        process_era5_land_orography_and_aspect(
            era5_geopotential_path=era5_geopotential_path,
            output_path=output_path,
            country=country,
            country_geometry=country_geometry,
            overwrite=overwrite,
            complevel=netcdf_compression_level,
        )
    )
else:
    print_step("ERA5 Land processing disabled")

for unit_name, input_tif in glo90_units.items():
    print_step(f"Starting GLO 90 processing unit: {unit_name}")
    input_tif = Path(input_tif)

    if not input_tif.exists():
        raise FileNotFoundError(f"Input file for unit '{unit_name}' was not found: {input_tif}")

    print_raster_info(input_tif, f"GLO 90 input for {unit_name}")

    orography_nc = output_path / f"glo90_orography_{country}_{unit_name}.nc"
    aspect_tif = output_path / f"glo90_aspect_{country}_{unit_name}.tif"
    aspect_nc = output_path / f"glo90_aspect_{country}_{unit_name}.nc"

    if run_glo90_orography_netcdf:
        outputs[f"glo90_orography_{unit_name}"] = raster_to_netcdf(
            input_tif=input_tif,
            output_nc=orography_nc,
            variable_name=orography_variable_name,
            geometry=country_geometry,
            overwrite=overwrite,
            complevel=netcdf_compression_level,
        )

    if run_glo90_aspect:
        compute_aspect(
            input_tif=input_tif,
            output_tif=aspect_tif,
            method=aspect_method,
            overwrite=overwrite,
            tile_size=aspect_tile_size,
        )

        outputs[f"glo90_aspect_{unit_name}"] = raster_to_netcdf(
            input_tif=aspect_tif,
            output_nc=aspect_nc,
            variable_name=aspect_variable_name,
            geometry=country_geometry,
            overwrite=overwrite,
            complevel=netcdf_compression_level,
        )

    print_step(f"Completed GLO 90 processing unit: {unit_name}")
    gc.collect()
    print_memory(f"Memory after {unit_name}")

print_step("Separate GLO 90 unit processing completed")
print("Generated intermediate files:")
for key, value in outputs.items():
    print(f"{key}: {value}")


[11:38:28] Chile country geometry loaded for country crop
The geometry was created by dissolving all polygons from REGIONES_v1.shp.
Chile geometry bounds in EPSG:4326: [-109.45491616  -56.53776582  -66.41559401  -17.49839934]
[11:38:28] Starting ERA5 Land orography and aspect processing
ERA5 Land input: ../DEMdata/geo_1279l4_0.1x0.1.grib2_v4_unpack.nc
ERA5 Land method: one Chile crop using the dissolved country geometry, no continental/islands split.
Before opening ERA5 Land geopotential: 1.22 GB used by the current Python process
[11:38:28] Opening ERA5 Land geopotential
Using ERA5 Land variable: z
Longitude range before normalization: 0.0000 to 359.9000
[11:38:28] Converting longitudes from 0..360 to -180..180
Longitude range after normalization: -180.0000 to 179.9000
Prepared spatial dataset bounds: lon -180.0000 to 179.9000, lat -90.0000 to 90.0000
Full ERA5 Land orography dimensions before country crop: {'latitude': 1801, 'longitude': 3600}
[11:38:28] Saving full ERA5 Land orograp

## Step 7. Merge the two GLO 90 NetCDF files at the end

The final GLO 90 merge is done only after the separate files have been written.

This keeps the most memory intensive work, especially aspect calculation, split into smaller files.


In [7]:
def open_for_merge(path, chunks=None):
    """Open one NetCDF file for merging and standardise coordinate names."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"NetCDF file not found: {path}")

    try:
        ds = xr.open_dataset(path, chunks=chunks)
    except ValueError:
        print("Dask chunking is not available. Opening without chunks.", flush=True)
        ds = xr.open_dataset(path)

    ds = rename_coordinates(ds)
    return ds


def merge_two_netcdfs(first_nc, second_nc, output_nc, variable_name, overwrite=True, chunks=None, complevel=4):
    """Merge two NetCDF files onto one outer latitude and longitude grid."""
    first_nc = Path(first_nc)
    second_nc = Path(second_nc)
    output_nc = Path(output_nc)
    output_nc.parent.mkdir(parents=True, exist_ok=True)

    if not should_write(output_nc, overwrite):
        print_step(f"Skipping existing merged NetCDF: {output_nc}")
        return output_nc

    print_step(f"Merging NetCDF files into: {output_nc.name}")
    print(f"First input: {first_nc}", flush=True)
    print(f"Second input: {second_nc}", flush=True)
    print_memory("Before opening NetCDF files for merge")

    ds1 = open_for_merge(first_nc, chunks=chunks)
    ds2 = open_for_merge(second_nc, chunks=chunks)

    if variable_name not in ds1.data_vars:
        raise ValueError(f"Variable '{variable_name}' not found in {first_nc}. Available variables: {list(ds1.data_vars)}")
    if variable_name not in ds2.data_vars:
        raise ValueError(f"Variable '{variable_name}' not found in {second_nc}. Available variables: {list(ds2.data_vars)}")

    print(f"First dataset dimensions: {dict(ds1.sizes)}", flush=True)
    print(f"Second dataset dimensions: {dict(ds2.sizes)}", flush=True)

    # combine_by_coords creates the outer coordinate grid and keeps values from both inputs.
    merged = xr.combine_by_coords([ds1, ds2], combine_attrs="override")
    merged = merged[[variable_name]]
    merged = drop_auxiliary_coordinates(merged)

    print(f"Merged dataset dimensions: {dict(merged.sizes)}", flush=True)
    print_memory("Before saving merged NetCDF")

    save_netcdf_compressed(merged, output_nc, complevel=complevel)

    ds1.close()
    ds2.close()
    del ds1, ds2, merged
    gc.collect()
    print_memory("After saving merged NetCDF and cleaning variables")

    return output_nc


if run_glo90_final_merge:
    continental_orography_nc = output_path / f"glo90_orography_{country}_continental.nc"
    islands_orography_nc = output_path / f"glo90_orography_{country}_islands.nc"
    continental_aspect_nc = output_path / f"glo90_aspect_{country}_continental.nc"
    islands_aspect_nc = output_path / f"glo90_aspect_{country}_islands.nc"

    merged_orography_nc = output_path / f"glo90_orography_{country}.nc"
    merged_aspect_nc = output_path / f"glo90_aspect_{country}.nc"

    outputs["glo90_orography_merged"] = merge_two_netcdfs(
        first_nc=continental_orography_nc,
        second_nc=islands_orography_nc,
        output_nc=merged_orography_nc,
        variable_name=orography_variable_name,
        overwrite=overwrite,
        chunks=merge_chunks,
        complevel=netcdf_compression_level,
    )

    outputs["glo90_aspect_merged"] = merge_two_netcdfs(
        first_nc=continental_aspect_nc,
        second_nc=islands_aspect_nc,
        output_nc=merged_aspect_nc,
        variable_name=aspect_variable_name,
        overwrite=overwrite,
        chunks=merge_chunks,
        complevel=netcdf_compression_level,
    )
else:
    print_step("Final merge disabled")

print_step("Workflow completed")
print("Generated files:")
for key, value in outputs.items():
    print(f"{key}: {value}")


[11:48:20] Merging NetCDF files into: glo90_orography_chile.nc
First input: ../DEMdata/chile/glo90_orography_chile_continental.nc
Second input: ../DEMdata/chile/glo90_orography_chile_islands.nc
Before opening NetCDF files for merge: 1.89 GB used by the current Python process
Dask chunking is not available. Opening without chunks.
Dask chunking is not available. Opening without chunks.
First dataset dimensions: {'latitude': 46848, 'longitude': 10554}
Second dataset dimensions: {'latitude': 9053, 'longitude': 36823}
Merged dataset dimensions: {'latitude': 46848, 'longitude': 47377}
Before saving merged NetCDF: 10.16 GB used by the current Python process
After saving merged NetCDF and cleaning variables: 1.89 GB used by the current Python process
[11:49:47] Merging NetCDF files into: glo90_aspect_chile.nc
First input: ../DEMdata/chile/glo90_aspect_chile_continental.nc
Second input: ../DEMdata/chile/glo90_aspect_chile_islands.nc
Before opening NetCDF files for merge: 1.89 GB used by the cu

## Notes for users

Use this notebook when both ERA5 Land and GLO 90 terrain variables are required by the ATLAS workflow.

For **ERA5 Land**, the workflow creates one single Chile product. It uses `REGIONES_v1.shp`, dissolves all regional polygons into one national Chile geometry in `EPSG:4326`, and crops the ERA5 Land orography and aspect outputs to that geometry. There is no `continental` or `islands` split for ERA5 Land.

For **GLO 90**, the full country raster can be too large to process safely. This is why `continental` and `islands` are processed separately and merged only at the end.

If aspect calculation is slow or memory intensive, keep:

```python
aspect_method = "auto"
aspect_tile_size = 2048
```

If memory is still high, reduce `aspect_tile_size` to `1024` or `512`.
